<a href="https://colab.research.google.com/github/ksenia-andreeva/kan-physics-recovery/blob/main/notebooks/kan_vs_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практика: Восстановление физического закона из данных**

Задача: сгенерировать данные затухающего гармонического осциллятора и восстановить закон движения.

# **Часть 1. Генерация данных**

1.1. Генерация траекторий численным интегрированием: координата *x(t)*, скорость *v(t)* и ускорение *a(t)* при **разных начальных условиях** и **разных константах** *k*, *c* - жесткости пружины и сопротивления среды.

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np

def rhs(t, y):
    x, v = y
    a = -(k/m)*x - (c/m)*v  # правая часть уравнения
    return [v, a]

all_data = []
m = 1.0  # масса маятника

for k in [2.0, 4.0, 6.0]: # цикл по жесткости и сопротивлению
    for c in [0.1, 0.5, 1.0]:

        for x0 in [1.0, 0.5, 2.0]: # цикл по начальным условиям
            for v0 in [0.0, 0.5, -0.5]:

                sol = solve_ivp(rhs, (0, 20), y0=[x0, v0], t_eval=np.linspace(0, 20, 500))
                # запуск численного интегрирования от t[0,20], 200 значений

                x, v = sol.y # извлечение из результата (sol) массив из координат и скоростей
                a = -(k/m)*x - (c/m)*v # вычисление ускорения

                # сохрание точек (x, v, a)
                for i in range(len(x)):
                    all_data.append([x[i], v[i], k, c, a[i]])

1.2. Формирование датасета из полученных данных с применением нормализации, так как приведенность данных к единому масштабу критически важна при обучении MLP (при обучении KAN - полезно, но не обзательно).

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

all_data = np.array(all_data) # список -> массив NumPy

# разделение значений на х (вход) и у (выход)
x = all_data[:, :4]   # (x, v, k, c)
y = all_data[:, 4]    # a

# разбивание данных на train/test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


# НОРМАЛИЗАЦИЯ ДАННЫХ - приведение к нулевому среднему и единичному стандартному отклонению
scaler_x = StandardScaler() # масштабирование входов
x_train_scaled = scaler_x.fit_transform(x_train)
x_test_scaled = scaler_x.transform(x_test)

scaler_y = StandardScaler() # масштабирование выходов (опционально)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print(f"Среднее x_train: {x_train_scaled.mean():.2f}")
print(f"Станд. откл. x_train: {x_train_scaled.std():.2f}")


# превращение данных в тензоры PyTorch, можно передавать MLP для обучения
x_train_t = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32).reshape(-1, 1)
x_test_t = torch.tensor(x_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32).reshape(-1, 1)

# сборка датасет для pykan
dataset = {
    'train_input': x_train_t,
    'train_label': y_train_t,
    'test_input': x_test_t,
    'test_label': y_test_t
}

print(f"Размер x_train: {x_train.shape}")
print(f"Размер y_train: {y_train.shape}")

# **Часть 2. Обучение KAN (без шума)**

2.1 Установка pykan


In [ ]:
!pip install git+https://github.com/KindXiaoming/pykan.git

2.2. Создание KAN: 4 входа, 2 скрытых нейрона, 1 выход, узлов сетки grid=6, кубические сплайны k=3

In [ ]:
from kan import KAN
from kan.utils import ex_round
import torch
torch.set_default_dtype(torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работает на устройстве: {device}")

# создание KAN
model_kan = KAN(width=[4, 2, 1], grid=6, k=3, seed=42, device=device)

print("Этап 1: обучение без регуляризации")
model_kan.fit(dataset, opt="LBFGS", steps=100, lamb=0.0)
model_kan.plot()

print("Этап 2: дообучение")
model_kan.fit(dataset, opt="LBFGS", steps=50, lamb=0.0)
model_kan.plot()

print("Этап 3: символическая регрессия")
model_kan.auto_symbolic()
ex_round(model_kan.symbolic_formula()[0][0],4)

Работает на устройстве: cpu
checkpoint directory created: ./model
saving model version 0.0
Этап 1: обучение без регуляризации


| train_loss: 3.42e-02 | test_loss: 3.48e-02 | reg: 1.92e+01 | : 100%|█| 100/100 [04:18<00:00,  2.59


saving model version 0.1
Этап 2: дообучение


| train_loss: 6.04e-02 | test_loss: 6.04e-02 | reg: 1.78e+01 | : 100%|█| 50/50 [02:08<00:00,  2.56s/


saving model version 0.2
Этап 3: символическая регрессия


# **Часть 3. Обучение MLP (без шума)**

In [ ]:

# 1. Описываем архитектуру нашей нейросети
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        # Первый полносвязный слой: вход (4) -> скрытый слой (16 нейронов)
        self.fc1 = nn.Linear(4, 16)
        # Второй полносвязный слой: скрытый (16) -> скрытый (16)
        self.fc2 = nn.Linear(16, 16)
        # Выходной слой: скрытый (16) -> выход (2 класса)
        self.fc3 = nn.Linear(16, 2)

    def forward(self, x):
        # Пропускаем данные через слои и используем функцию активации ReLU
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # На последнем слое обычно не применяют ReLU
        x = self.fc3(x)
        return x

# 3. Создаем случайные входные данные (батч из 1 примера, 4 признака)
input_data = torch.randn(1, 4)

# 4. Получаем предсказание сети
output = model(input_data)
print("Результат работы сети:", output)


import torch.nn as nn

# запись архитектуры MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# создание MLP
model_mlp = MLP
print(model_mlp)